# Hear what audio ducking changes in a video edit

Listen to two exports of the same 12-second mix. A low chord plays throughout. A higher tone stands in for the dialogue track during seconds 2–4 and 6–8. In one export the music stays at fixed gain. In the other, the higher tone triggers a compressor on the music. Can you hear when the chord comes back?

The examples come from the Valmera renderer using synthetic inputs. Run all cells to hear both files at the same gain, plot the music level, and measure five listening windows. The final cell accepts two exports of the same fixture from another editor. [Open the original Valmera example](https://valmera.io/tools/add-music-to-video?utm_source=colab&utm_medium=notebook&utm_campaign=audio_ducking).

## Load the two reference mixes

In Colab, run the notebook with the default CPU runtime. Locally it uses Python, NumPy, Matplotlib, IPython and FFmpeg. When you extract the dataset beside the notebook, the reference files load from disk. Otherwise they download from Valmera. Their SHA-256 hashes identify the exact two published mixes.

In [ ]:
import hashlib
import shutil
import subprocess
import urllib.request
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

rate = 48_000
seconds = 12
expected_hash = {
    "without-ducking.m4a": "8ea566b099b2702f3ba4dbdb8ded048e48315c0edc315cd0c76428b6742c8910",
    "with-ducking.m4a": "8b0bc0ab6d8c9f859b3c38ed16f7c8779ee1b21f72518bfda16226f8aba8a180",
}
def reference_file(name):
    locations = [Path(name)]
    if Path("/kaggle/input").is_dir():
        locations.extend(Path("/kaggle/input").rglob(name))
    for path in locations:
        if path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == expected_hash[name]:
            return path
    data = urllib.request.urlopen("https://valmera.io/examples/music-ducking/" + name, timeout=30).read()
    if hashlib.sha256(data).hexdigest() != expected_hash[name]:
        raise ValueError(f"The reference file changed: {name}")
    path = Path("valmera-ducking-reference") / name
    path.parent.mkdir(exist_ok=True)
    path.write_bytes(data)
    return path

ffmpeg = shutil.which("ffmpeg")
if not ffmpeg:
    import imageio_ffmpeg  # Optional local alternative to an FFmpeg executable on PATH.
    ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()

def decode(path):
    command = [ffmpeg, "-v", "error", "-i", str(path), "-map", "0:a:0", "-t", str(seconds),
               "-ac", "1", "-ar", str(rate), "-f", "f32le", "-"]
    pcm = np.frombuffer(subprocess.check_output(command), dtype="<f4")
    if len(pcm) < rate * seconds:
        raise ValueError(f"{path} has less than 12 seconds of audio. Check the export range.")
    return pcm[:rate * seconds]

fixed, ducked = [decode(reference_file(n)) for n in ("without-ducking.m4a", "with-ducking.m4a")]

## Listen at one volume

Play the fixed mix first. The higher note arrives at second 2. Now play the ducked mix at the same device volume. Listen to the lower chord underneath that note and again as the note stops at second 4. Playback below preserves the level difference between the files.

In [ ]:
for label, samples in [("Fixed music gain", fixed), ("Ducking enabled", ducked)]:
    print(label)
    display(Audio(samples, rate=rate, normalize=False))

## See how far the music falls

We measure energy from 180 to 500 Hz, where the three notes of the chord sit. The 880 Hz trigger is outside this band. A meter for the entire mix would also count the trigger, making the change to the chord harder to see. Each point compares the same 100 ms window of the two exports, with a Hann taper and a 50 ms step. Zero means equal band energy. Negative values mean quieter music.

In [ ]:
def band_energy(samples):
    tapered = samples * np.hanning(len(samples))
    frequencies = np.fft.rfftfreq(len(samples), 1 / rate)
    selected = (frequencies >= 180) & (frequencies <= 500)
    spectrum = np.fft.rfft(tapered)
    return np.sum(np.abs(spectrum[selected]) ** 2)

def difference_db(off, on):
    off_power, on_power = band_energy(off), band_energy(on)
    if min(off_power, on_power) <= 1e-12:
        return np.nan
    return 10 * np.log10(on_power / off_power)

def trace(off, on):
    width, hop = rate // 10, rate // 20
    starts = np.arange(0, len(off) - width + 1, hop)
    levels = [difference_db(off[i:i+width], on[i:i+width]) for i in starts]
    return (starts + width / 2) / rate, levels

def show_trace(off, on, title):
    time, levels = trace(off, on)
    fig, ax = plt.subplots(figsize=(9, 3.3), layout="constrained")
    for start in (2, 6):
        ax.axvspan(start, start + 2, color="#d9e8ee", label="Trigger tone" if start == 2 else None)
    ax.axhline(0, color="#78838a", linewidth=0.8)
    ax.plot(time, levels, color="#92264d", linewidth=2, label="Music band change")
    ax.set(xlim=(0, seconds), xlabel="Time in the export (s)", ylabel="Ducking on / off (dB)", title=title)
    ax.legend(frameon=False, loc="lower right")
    plt.show()
    return fig

chart = show_trace(fixed, ducked, "How the background chord responds to the trigger")

## Check five moments in the export

Both ranges inside the higher note are measured after the compressor has had time to respond. The recovery window starts a full second after the first note stops. These readings describe the supplied test tones. They do not rate speech clarity or establish a preferred amount of ducking for dialogue.

In [ ]:
windows = [
    ("Before the trigger", 0.8, 1.6), ("Inside first tone", 3.0, 3.8),
    ("Between tones", 5.0, 5.8), ("Inside second tone", 7.0, 7.8),
    ("After both tones", 10.0, 10.8),
]
def measure(off, on):
    results = []
    for label, start, end in windows:
        section = slice(round(start * rate), round(end * rate))
        level = difference_db(off[section], on[section])
        results.append(level)
        print(f"{label:20} {start:4.1f}–{end:4.1f} s   {level:+6.2f} dB")
    return results

reference_db = measure(fixed, ducked)

## Compare exports from your editor

Import the [12-second video with the trigger tone](https://valmera.io/examples/music-ducking/source-with-tones.mp4) and the [music bed WAV](https://valmera.io/examples/music-ducking/music-bed.wav). Align both at zero. Keep the music fader, duration and master settings the same across two exports. Use the audio from the video as the trigger and apply ducking to the music in the second export only. Disable automatic loudness matching or mastering if your editor permits it.

Place the files beside this notebook or in your notebook runtime. Enter both paths below and run the cell. A dip before the first tone often points to an alignment or gain difference. If the music stays low in the gap, listen for a slow release. For real dialogue, repeat the listening check with actual words, breaths and pauses. This frequency test applies specifically to the supplied synthetic tracks.

In [ ]:
off_path = ""  # e.g. "/content/my-fixed-mix.mp4"
on_path = ""   # e.g. "/content/my-ducked-mix.mp4"
if bool(off_path) != bool(on_path):
    raise ValueError("Provide both exports to compare the effect.")
if off_path and on_path:
    my_fixed, my_ducked = decode(off_path), decode(on_path)
    measure(my_fixed, my_ducked)
    show_trace(my_fixed, my_ducked, "Your editor’s two exports of this fixture")

[Valmera’s original settings, file hashes and measurements](https://valmera.io/examples/music-ducking/evidence.json) · [Edit a recording with Valmera](https://valmera.io/docs/getting-started?utm_source=colab&utm_medium=notebook&utm_campaign=audio_ducking)